# FEM-FEM Inverse — Per-Patient Viewer

Browse the inverse-bioheat result for **any patient**: the recovered tumour (z, r, lateral),
and a 2×2 anterior-view render — **IR measured (input)**, **FEM forward (prediction at the
recovered tumour)**, **residual |FEM−IR|**, and the **recovered tumour location**.

Uses the **corrected** boundary conditions (normal-based chest wall, fixed 2026-06-17) and the
BC-fixed cohort (`results/cohort_BCfix_ALL.csv`). Each patient needs one NeRF geometry load
(~30 s) + one FEM solve, so view on demand rather than all 122 at once.


In [ ]:
import os
os.environ["PYVISTA_OFF_SCREEN"] = "true"
import numpy as np, pandas as pd
from pathlib import Path
import pyvista as pv
pv.OFF_SCREEN = True
import run_cohort as R
import mukhmetov_recover as M
from IPython.display import Image, display

dataset, patients = R.build_dataset()
encoder, mlp = R.load_models()
id2idx = {p['id']: i for i, p in enumerate(patients)}

COH = Path('results/cohort_BCfix_ALL.csv')   # corrected-BC cohort
coh = pd.read_csv(COH).set_index('patient_id')
print(f'loaded {len(coh)} patient results from {COH.name}')


## 1. Summary table — all patients

In [ ]:
def rail_kind(r):
    return 'upper rail' if r >= 39.9 else ('lower rail' if r <= 7.05 else 'interior')

summ = coh.copy()
summ['rail']      = summ['r_hat_mm'].apply(rail_kind)
summ['FEM_fail']  = (summ['final_cost'] > 100) | (summ['fea_mean_residual'] > 20)
cols = ['label','x0_mm','y0_mm','z_hat_mm','r_hat_mm','final_cost','fea_mean_residual','rail','FEM_fail']
print('rail / class breakdown (valid patients):')
print(summ[~summ.FEM_fail].groupby(['label','rail']).size())
print(f"\nFEM failures (excluded): {summ.FEM_fail.sum()}  -> {sorted(summ[summ.FEM_fail].index.tolist())}")
summ[cols].sort_index()


## 2. Per-patient viewer

Set `PATIENT` and run. Panels:
1. **IR (input)** — measured thermogram (anterior skin).
2. **FEM forward** — predicted skin temperature at the recovered tumour.
3. **Residual |FEM−IR|** — what the homogeneous model cannot reproduce.
4. **Tumour location** — recovered tumour (cyan sphere) on the breast.

The printout also gives the **skin-only residual** (the honest fit metric — the CSV's
`fea_mean_residual` is over the whole surface and is inflated by the 37 °C posterior chest wall).


In [ ]:
def _surf(geo):
    f = geo['faces'].astype(np.int64)
    faces = np.hstack([np.full((len(f),1),3,dtype=np.int64), f]).ravel()
    return pv.PolyData(geo['surface_pts'].astype(float), faces)

def _front_cam(surf):
    b = surf.bounds
    cx, cy, cz = (b[0]+b[1])/2, (b[2]+b[3])/2, (b[4]+b[5])/2
    camz = b[4] - 2.2*max(b[1]-b[0], b[3]-b[2])   # anterior (-Z) view, superior up
    return [(cx, cy, camz), (cx, cy, cz), (0, -1, 0)]

def view_patient(pid):
    row = coh.loc[pid]; idx = id2idx[pid]
    x0, y0, z, r = (float(row[k]) for k in ('x0_mm','y0_mm','z_hat_mm','r_hat_mm'))
    geo, msh = M.load_geo_and_mesh(dataset, idx, encoder, mlp, R.RESULTS_DIR/pid, mesh_suffix='syn')
    T_fem = np.asarray(M.fem_surface(msh, geo, x0, y0, z, r), float)
    T_ir  = np.asarray(geo['T_measured'], float)
    res   = np.abs(T_fem - T_ir)
    skin  = ~R.chest_wall_mask_from_pts(geo['surface_pts'], geo['vertex_normals'])
    s = _surf(geo); s['IR']=T_ir; s['FEM']=T_fem; s['RES']=res
    cam = _front_cam(s)
    cir = [float(np.percentile(T_ir[skin],2)),  float(np.percentile(T_ir[skin],98))]
    cfe = [float(np.percentile(T_fem[skin],2)), float(np.percentile(T_fem[skin],98))]
    pl = pv.Plotter(off_screen=True, shape=(2,2), window_size=(1400,1400), border=True)
    pl.subplot(0,0); pl.add_mesh(s.copy(),scalars='IR', cmap='inferno',clim=cir,scalar_bar_args=dict(title='IR (C)'));  pl.add_text('1. IR (input)',font_size=10);  pl.camera_position=cam
    pl.subplot(0,1); pl.add_mesh(s.copy(),scalars='FEM',cmap='inferno',clim=cfe,scalar_bar_args=dict(title='FEM (C)')); pl.add_text('2. FEM forward',font_size=10); pl.camera_position=cam
    pl.subplot(1,0); pl.add_mesh(s.copy(),scalars='RES',cmap='turbo',scalar_bar_args=dict(title='|FEM-IR| C'));         pl.add_text('3. Residual',font_size=10);   pl.camera_position=cam
    pl.subplot(1,1); pl.add_mesh(s.copy(),scalars='IR',cmap='inferno',clim=cir,opacity=0.5)
    pl.add_mesh(pv.Sphere(radius=max(r,3.0),center=(x0,y0,z)),color='cyan',opacity=0.6)
    pl.add_text(f'4. Tumour z={z:.0f} r={r:.0f}',font_size=10); pl.camera_position=cam
    out = f'results/patient_grids/{pid}_grid.png'
    Path(out).parent.mkdir(parents=True, exist_ok=True)
    pl.screenshot(out); pl.close()
    print(f"{pid} ({row.label}):  z_hat={z:.1f} mm  r_hat={r:.1f} mm  cost={row.final_cost:.2f}")
    print(f"  skin-only residual = {res[skin].mean():.2f} C   (CSV all-surface = {row.fea_mean_residual:.2f} C, inflated by posterior)")
    display(Image(out))


In [ ]:
PATIENT = 'Patient_1'
view_patient(PATIENT)


## 3. Why the radius rails — cost vs radius (optional)

For a chosen patient, sweep the bilateral cost `J(r)` at the recovered depth. A nearly-flat
curve (≈0.7 % over r∈[5,40]) means the radius is **unidentifiable** — the rail is the
depth–size degeneracy, not the optimiser (see fig12).

In [ ]:
import matplotlib.pyplot as plt

def cost_vs_radius(pid, r_list=(5,8,11,14,17,20,24,28,32,36,40)):
    row = coh.loc[pid]; idx = id2idx[pid]
    x0, y0, z = float(row.x0_mm), float(row.y0_mm), float(row.z_hat_mm)
    geo, msh = M.load_geo_and_mesh(dataset, idx, encoder, mlp, R.RESULTS_DIR/pid, mesh_suffix='syn')
    sp, Tm = geo['surface_pts'], np.asarray(geo['T_measured'], float)
    sk  = ~R.chest_wall_mask_from_pts(sp, geo['vertex_normals'])
    mir, x_mid, _ = M.build_mirror_map(geo)
    side = (sp[:,0] > x_mid) if x0 > x_mid else (sp[:,0] < x_mid)
    aff = sk & side
    A_tgt = (Tm[aff] - Tm[mir[aff]]).astype(np.float64)
    J = []
    for r in r_list:
        Tc = np.asarray(M.fem_surface(msh, geo, x0, y0, z, r), float)
        Ac = Tc[aff] - Tc[mir[aff]]
        J.append(float(np.mean((Ac - A_tgt)**2)))
    J = np.array(J)
    fig, ax = plt.subplots(figsize=(6,4))
    ax.plot(r_list, J, 'o-', color='#C44E52')
    ax.set_xlabel('radius r (mm)'); ax.set_ylabel('bilateral cost J')
    ax.set_title(f'{pid}: J(r) at z={z:.0f} mm — variasi {(J.max()-J.min())/J.mean()*100:.1f}%')
    plt.show()
    print(f'argmin r = {r_list[int(J.argmin())]} mm   (40=upper rail)')

cost_vs_radius(PATIENT)
